In [19]:
from concurrent.futures import ThreadPoolExecutor
from contextlib import closing
from os import environ
from pathlib import Path
from shutil import which
from time import perf_counter

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from checkpointing import SPATIAL_CHECKPOINT_DIR
from model import FireForecast, MemoryAdapter, RolloutSpatialCache
from prepare_data import has_fire_observation

torch.set_num_threads(4)


In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_dir=Path("data")
learning_rate=3e-5
low_vram=False
gpu_cache_gb=4.0
use_bf16=device.type=="cuda" and torch.cuda.is_bf16_supported()
history_days=None
rollout_steps=16
loss_decay=0.9
synthetic_gap_probability=0.15  # Chance to start a gap on an eligible training day. Set to 0 to disable.
synthetic_gap_lengths=(1, 3)
labels=("active", "burned", "new")

class FireDataset(Dataset):
    def __init__(self, files):
        self.files=files
        self.clear_cache()

    def __len__(self):
        return len(self.files)-1

    def read_day(self, index):
        if self.cache is not None:
            return self.cache[index]
        with np.load(self.files[index]) as data:
            day={name: data[name] for name in ("image", "date", *labels)}
        image=day["image"]
        valid=np.isfinite(image)
        day["observation_available"]=has_fire_observation(valid)
        day["image"]=np.concatenate([np.where(valid, image, 0), valid], axis=0)
        return day

    def prepare_cache(self):
        if self.cache is None:
            self.cache=[self.read_day(i) for i in range(len(self.files))]
            self.cpu_images=np.stack([day["image"] for day in self.cache])
            for index, day in enumerate(self.cache):
                day["image"]=self.cpu_images[index]

    def load_cache(self):
        self.prepare_cache()
        self.images=torch.from_numpy(self.cpu_images).to(
            device=device, dtype=torch.bfloat16 if use_bf16 else torch.float32, copy=True
        )
        self.original_images=None if low_vram else self.images.clone()
        self.dates=torch.tensor([day["date"].item() for day in self.cache], device=device, dtype=torch.float32)
        self.targets={name: torch.as_tensor(np.stack([day[name] for day in self.cache]), device=device, dtype=torch.float32) for name in labels}

    def mask_observation(self, index):
        # Clear VIIRS and fire-map channels in both the values and validity mask.
        frame=self.images[index].unflatten(0, (2, 29))
        frame[:, :8].zero_()
        frame[:, 27:29].zero_()

    def clear_cache(self):
        self.cache=self.cpu_images=self.images=self.original_images=self.dates=self.targets=None

    def __getitem__(self, index):
        stop=index+1
        start=0 if history_days is None else max(0, stop-history_days)
        if self.images is not None:
            history=self.images[start:stop]
            dates=self.dates[start:stop]
            targets={name: values[stop] for name, values in self.targets.items()}
        else:
            days=[self.read_day(i) for i in range(start, stop)]
            history=torch.from_numpy(np.stack([day["image"] for day in days]))
            dates=torch.tensor([day["date"].item() for day in days], dtype=torch.float32)
            target=self.read_day(stop)
            targets={name: torch.tensor(target[name], dtype=torch.float32) for name in labels}
        return history, dates-dates[-1], targets


def sample_synthetic_gaps(observed_days):
    masked_days=set()
    origin=1
    while origin<len(observed_days):
        # Keep a usable day before each gap so the SNN has state to carry.
        lengths=[
            length for length in synthetic_gap_lengths
            if origin+length<=len(observed_days) and all(observed_days[origin:origin+length])
        ]
        if observed_days[origin-1] and lengths and np.random.random()<synthetic_gap_probability:
            length=int(np.random.choice(lengths))
            masked_days.update(range(origin, origin+length))
            origin+=length+1
        else:
            origin+=1
    return masked_days


def prefetch_fires(loaders, order):
    with ThreadPoolExecutor(max_workers=1) as pool:
        pending=pool.submit(loaders[order[0]].dataset.prepare_cache)
        try:
            for position, index in enumerate(order):
                pending.result()
                if position+1<len(order):
                    pending=pool.submit(loaders[order[position+1]].dataset.prepare_cache)
                yield loaders[index].dataset
        finally:
            try:
                pending.result()
            finally:
                for index in order:
                    loaders[index].dataset.clear_cache()


In [21]:
train_fires=Path("train_fires.txt").read_text().splitlines()
val_fires=Path("val_fires.txt").read_text().splitlines()

loader_options=dict(num_workers=0, pin_memory=False)

train_loaders=[DataLoader(FireDataset(sorted((data_dir/fire).glob("*.npz"))), batch_size=None, shuffle=False, **loader_options) for fire in train_fires]
val_loaders=[DataLoader(FireDataset(sorted((data_dir/fire).glob("*.npz"))), batch_size=None, shuffle=False, **loader_options) for fire in val_fires]
train_data=train_loaders[0].dataset

In [22]:
memory=MemoryAdapter()
network=FireForecast(input_channels=train_data[0][0].shape[1],memory=memory, low_vram=low_vram, gpu_cache_gb=gpu_cache_gb).to(device)
optimizer=torch.optim.AdamW(network.parameters(), lr=learning_rate, weight_decay=1e-4)

if device.type=="cuda":
    torch.backends.cudnn.benchmark=True

# Inductor needs C and C++ compilers for this graph.
c_compiler=which(environ["CC"]) if environ.get("CC") else which("gcc") or which("clang")
cpp_compiler=which(environ.get("CXX") or "g++")
if device.type=="cuda" and c_compiler and cpp_compiler:
    torch._dynamo.config.cache_size_limit=32
    network.encoder._encode_day=torch.compile(network.encoder._encode_day, fullgraph=False, dynamic=True)
    network.encoder._aggregate_skip=torch.compile(network.encoder._aggregate_skip, fullgraph=False, dynamic=True)
    network.decoder.forward=torch.compile(network.decoder.forward, fullgraph=False, dynamic=False)
    network.output.forward=torch.compile(network.output.forward, fullgraph=False, dynamic=False)

In [23]:
source_epoch=41
epochs=50
patience=7
min_delta=0.0001
noimprove=0
best_val_loss=float("inf")
stopping_loss=float("inf")
losses=[]

checkpoint_dir=SPATIAL_CHECKPOINT_DIR
checkpoint_dir.mkdir(parents=True, exist_ok=True)
resume_path=checkpoint_dir / "latest.pt"
initial_path=checkpoint_dir / f"epoch_{source_epoch:03d}.pt"
rollout_config={
    "rollout_steps": rollout_steps,
    "history_days": history_days,
    "loss_decay": loss_decay,
    "feedback": "probability",
    "memory_updates": "observed_issuance_only",
    "synthetic_gap_probability": synthetic_gap_probability,
    "synthetic_gap_lengths": synthetic_gap_lengths,
}
if resume_path.exists():
    checkpoint=torch.load(resume_path, map_location="cpu", weights_only=True)
    if checkpoint.get("source_epoch")!=source_epoch or checkpoint["epoch"]<source_epoch:
        raise ValueError(f"{resume_path} does not belong to the epoch-{source_epoch} fine-tuning run.")
    network.load_state_dict(checkpoint["model"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer"])
    for group in optimizer.param_groups:
        group["lr"]=learning_rate
    start_epoch=checkpoint["epoch"]
    if all(checkpoint.get(key)==value for key, value in rollout_config.items()):
        best_val_loss=checkpoint["best_val_loss"]
        stopping_loss=checkpoint["stopping_loss"]
        noimprove=checkpoint["noimprove"]
        losses=checkpoint["losses"]
    print(f"Resuming {resume_path}; next spatial training epoch {start_epoch+1}.")
else:
    checkpoint=torch.load(initial_path, map_location="cpu", weights_only=True)
    if checkpoint.get("epoch")!=source_epoch:
        raise ValueError(f"Expected completed epoch {source_epoch} in {initial_path}.")
    network.load_state_dict(checkpoint["model"], strict=True)
    start_epoch=source_epoch
    print(f"Loaded {initial_path} with a fresh optimizer. Next epoch {start_epoch+1}.")
del checkpoint

Loaded /home/neptunium/AI/FireForecaster/data/checkpoints/spatial/epoch_041.pt with a fresh optimizer. Next epoch 42.


In [24]:
for epoch in range(start_epoch, epochs):
    if noimprove>=patience:
        break
    epoch_started=perf_counter()
    print(f"Epoch {epoch+1}/{epochs}:")
    for phase, loaders in (("train", train_loaders), ("val", val_loaders)):
        training=phase=="train"
        network.train(training)
        total_loss=torch.zeros((), device=device)
        samples=0
        order=np.random.permutation(len(loaders)) if training else range(len(loaders))

        forecast_steps=sum(
            min(rollout_steps, remaining)
            for loader in loaders
            for remaining in range(1, len(loader.dataset)+1)
        )
        with (
            tqdm(total=forecast_steps, desc=f"Epoch {epoch+1} {phase}", unit="step") as progress,
            closing(prefetch_fires(loaders, order)) as fires,
            torch.set_grad_enabled(training),
        ):
            for fire_number, dataset in enumerate(fires, start=1):
                progress.set_postfix_str(f"fire {fire_number}/{len(loaders)}", refresh=False)
                memory.reset_memory()
                try:
                    dataset.load_cache()
                    masked_days=set()
                    if training and synthetic_gap_probability:
                        observations=[day["observation_available"] for day in dataset.cache[:-1]]
                        masked_days=sample_synthetic_gaps(observations)
                    for origin in range(len(dataset)):
                        masked=origin in masked_days
                        if masked:
                            # Past gaps stay in the history seen from later origins.
                            dataset.mask_observation(origin)
                        observation_available=dataset.cache[origin]["observation_available"] and not masked
                        steps=min(rollout_steps, len(dataset)-origin)
                        weight_sum=sum(loss_decay**lead for lead in range(steps))
                        if training:
                            optimizer.zero_grad(set_to_none=True)
                        changed_days=[]
                        spatial_cache=RolloutSpatialCache(network.encoder, capacity=origin+steps) if history_days is None else None
                        try:
                            for step in range(steps):
                                history, dates, targets=dataset[origin+step]
                                history=history.unsqueeze(0)
                                dates=dates.unsqueeze(0).to(device)
                                targets={name: target.unsqueeze(0).to(device) for name, target in targets.items()}

                                memory.detach_memory()
                                write_memory=step==0 and observation_available
                                with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):
                                    active, burned, newly_burned=network(
                                        history, dates, spatial_cache=spatial_cache, update_memory=write_memory
                                    )
                                loss=F.binary_cross_entropy(active, targets["active"])+ F.binary_cross_entropy(burned, targets["burned"])+ 0.5*F.binary_cross_entropy(newly_burned, targets["new"])
                                weight=loss_decay**step/weight_sum
                                loss=loss*weight
                                if training:
                                    loss.backward()
                                total_loss+=loss.detach()
                                samples+=weight

                                if step+1<steps:
                                    target_day=origin+step+1
                                    changed_days.append(target_day)
                                    with torch.no_grad():
                                        dataset.images[target_day].copy_(dataset.images[origin])
                                        dataset.images[target_day, 27:28].copy_(active.detach()[0])
                                        dataset.images[target_day, 28:29].copy_(burned.detach()[0])
                                        dataset.images[target_day, 56:58]=1
                                del history, dates, targets, active, burned, newly_burned, loss
                                progress.update(1)
                            if training:
                                if spatial_cache is not None:
                                    spatial_cache.backward()
                                torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
                                optimizer.step()
                        finally:
                            if spatial_cache is not None:
                                spatial_cache.clear()
                            for day in changed_days:
                                original=dataset.original_images[day] if dataset.original_images is not None else torch.from_numpy(dataset.cache[day]["image"])
                                dataset.images[day].copy_(original)
                            changed_days.clear()
                finally:
                    memory.reset_memory()
                    dataset.clear_cache()

        average_loss=(total_loss/samples).item()
        print(f"{phase} loss: {average_loss:.6f}")
        if phase=="val":
            val_loss=average_loss

    losses.append(val_loss)
    improved=val_loss<best_val_loss
    if improved:
        best_val_loss=val_loss
    if val_loss<stopping_loss-min_delta:
        stopping_loss=val_loss
        noimprove=0
    else:
        noimprove+=1
        print(f"Early stopping: {noimprove}/{patience} epochs without improvement.")

    checkpoint={
        "epoch": epoch+1,
        "source_epoch": source_epoch,
        **rollout_config,
        "model": network.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "stopping_loss": stopping_loss,
        "noimprove": noimprove,
        "losses": losses,
    }
    paths=[resume_path]
    if improved:
        paths.append(checkpoint_dir / "best.pt")
    if (epoch+1)%5==0:
        paths.append(checkpoint_dir / f"epoch_{epoch+1:03d}.pt")
    for path in paths:
        torch.save(checkpoint, path.with_suffix(".tmp"))
        path.with_suffix(".tmp").replace(path)
    del checkpoint
    print(f"Saved {resume_path} after epoch {epoch+1}.")

    print(f"Epoch {epoch+1} took {(perf_counter()-epoch_started)/60:.1f} minutes.")
    if noimprove>=patience:
        print("Early stopped.")
        break


Epoch 42/50:


Epoch 42 train:   0%|          | 9/11959 [00:02<32:09,  6.19step/s, fire 1/96]  [W1115 16:20:14.230724121 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 455081984 bytes (free: 279969792, total: 12346195968).
[W1115 16:20:14.264272438 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 369098752 bytes (free: 183500800, total: 12346195968).
Epoch 42 train:   0%|          | 40/11959 [00:04<16:24, 12.11step/s, fire 1/96][W1115 16:20:16.742453812 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 364904448 bytes (free: 112197632, total: 12346195968).
[W1115 16:20:16.922602344 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 364904448 bytes (free: 252706816, total: 12346195968).
Epoch 42 train:   0%|          | 40/11959 [00:05<25:31,  7.78step/s, fire 1/96]


OutOfMemoryError: CUDA out of memory. Tried to allocate 348.00 MiB. GPU 0 has a total capacity of 11.50 GiB of which 241.00 MiB is free. Including non-PyTorch memory, this process has 11.25 GiB memory in use. Of the allocated memory 8.76 GiB is allocated by PyTorch, and 2.26 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)